In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("teste_koin.default.gold_orders_customers")

df_region_metrics = (
    df_gold
    .groupBy(
        "customer_state",
        "customer_city"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("order_amount"), 2).alias("total_revenue"),
        F.round(F.avg("order_amount"), 2).alias("average_ticket"),
        F.countDistinct("customer_id").alias("total_customers")
    )
    .orderBy(F.col("total_revenue").desc())
)

(
    df_region_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("teste_koin.default.gold_region_metrics")
)

In [0]:
%sql
select * from teste_koin.default.gold_region_metrics